# SCIKIT-LEARN MASTERY — NOTEBOOK 2
## Chapters 6–10: Categorical Encoding → Missing Value Imputation → Feature Generation → Custom Transformers → ColumnTransformer

Continuing directly from Notebook 1. This notebook builds toward your first genuine mixed-type preprocessing pipeline.

### SCIKIT-LEARN MASTERY PROGRESS
- [x] Scikit-learn Ecosystem, Estimator API, Data Representation, Train/Test Split, Scaling *(Notebook 1)*
- [x] Categorical Encoding
- [x] Missing Value Imputation
- [x] Feature Generation
- [x] Custom Transformers
- [x] ColumnTransformer


In [1]:
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt
np.random.seed(42)


---
# CHAPTER 6 — CATEGORICAL ENCODING

## 1. Concept
Scikit-learn's algorithms operate on numeric arrays — categorical (string/object) columns must be converted to numbers before fitting. **How** you do this conversion matters enormously, because a naive encoding can silently fabricate false mathematical relationships (order, distance) that don't exist in the original categories.

## 2. The Three Main Encoders

| Encoder | Output | Use for | Danger |
|---|---|---|---|
| `OneHotEncoder` | One binary column per category | **Feature columns**, unordered categories | High-cardinality columns explode dimensionality |
| `OrdinalEncoder` | One integer column, arbitrary/specified order | **Feature columns** with genuine order (e.g., "Low/Medium/High") | Implies numeric distance between categories — wrong for unordered data |
| `LabelEncoder` | One integer column, arbitrary order | **Target `y` only**, never feature columns | Same false-order problem as `OrdinalEncoder`, but historically misused on features |


In [2]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder

df_cat = pd.DataFrame({
    "color": ["red", "blue", "green", "blue", "red"],
    "size": ["small", "medium", "large", "medium", "small"],
})

# OneHotEncoder -- for UNORDERED categorical features
ohe = OneHotEncoder(sparse_output=False)
color_encoded = ohe.fit_transform(df_cat[["color"]])
print("OneHotEncoder output shape:", color_encoded.shape, " (one column per category)")
print("Categories learned:", ohe.categories_)
print(pd.DataFrame(color_encoded, columns=ohe.get_feature_names_out(["color"])))


OneHotEncoder output shape: (5, 3)  (one column per category)
Categories learned: [array(['blue', 'green', 'red'], dtype=object)]
   color_blue  color_green  color_red
0         0.0          0.0        1.0
1         1.0          0.0        0.0
2         0.0          1.0        0.0
3         1.0          0.0        0.0
4         0.0          0.0        1.0


In [3]:
# OrdinalEncoder -- for GENUINELY ORDERED categorical features (specify the order explicitly!)
size_order = [["small", "medium", "large"]]   # explicit order -- CRITICAL, don't leave this to alphabetical default
ordinal_enc = OrdinalEncoder(categories=size_order)
size_encoded = ordinal_enc.fit_transform(df_cat[["size"]])
print("OrdinalEncoder output (small=0, medium=1, large=2):")
print(size_encoded.ravel())


OrdinalEncoder output (small=0, medium=1, large=2):
[0. 1. 2. 1. 0.]


### Line-by-Line Explanation
- `OneHotEncoder(sparse_output=False)` — by default, `OneHotEncoder` returns a **sparse** matrix (memory-efficient for high-cardinality columns, Chapter 62); `sparse_output=False` forces a dense NumPy array here purely for readable display.
- `ohe.categories_` — the fitted attribute listing exactly which categories were learned, in which order — essential to inspect, since this determines the meaning of each output column.
- `ohe.get_feature_names_out(["color"])` — produces genuinely interpretable output column names (`color_blue`, `color_green`, `color_red`) rather than anonymous `x0_0`, `x0_1`, `x0_2` — critical for later feature-importance interpretation (Chapter 59's dedicated focus).
- `OrdinalEncoder(categories=[["small", "medium", "large"]])` — **explicitly** specifying the category order is essential; without it, `OrdinalEncoder` defaults to sorting categories **alphabetically**, which for "small/medium/large" would incorrectly order them as large=0, medium=1, small=2 — silently wrong for any genuinely ordinal column.

### `handle_unknown` — Handling Categories Never Seen During Training


In [4]:
ohe_unknown = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
ohe_unknown.fit(df_cat[["color"]])

new_data = pd.DataFrame({"color": ["red", "purple"]})   # "purple" was NEVER seen during fit
result = ohe_unknown.transform(new_data)
print("Transform with an unseen category ('purple'):")
print(result)
print("\n'purple' -> all-zero row (no error!) because handle_unknown='ignore'")

# WITHOUT handle_unknown="ignore" (the default), this would raise an error:
ohe_strict = OneHotEncoder(sparse_output=False)   # default handle_unknown="error"
ohe_strict.fit(df_cat[["color"]])
try:
    ohe_strict.transform(new_data)
except ValueError as e:
    print("\nWithout handle_unknown='ignore', this raises:")
    print(str(e)[:150])


Transform with an unseen category ('purple'):
[[0. 0. 1.]
 [0. 0. 0.]]

'purple' -> all-zero row (no error!) because handle_unknown='ignore'

Without handle_unknown='ignore', this raises:
Found unknown categories ['purple'] in column 0 during transform


### Parameter Explanation
- `handle_unknown="ignore"` — at transform time, any category not seen during `.fit()` produces an all-zero row instead of raising an error. **Essential for production**: real-world categorical data (a new product SKU, a new city) will eventually contain values your training data never saw, and a production inference pipeline that crashes on this is a serious reliability problem.
- `handle_unknown="error"` (the default) — appropriate during development/EDA when you specifically *want* to be alerted to unexpected categories, since it can also reveal a genuine data-quality issue (a typo, a merge bug) rather than a legitimately new category.
- `min_frequency` / `max_categories` (newer Scikit-learn parameters) — group rare categories into a single "infrequent" bucket, directly implementing the Chapter 70 (Matplotlib course)-style top-N-plus-Other pattern at the encoding level for high-cardinality columns.

### Why `LabelEncoder` Should Not Encode Feature Columns


In [5]:
# LabelEncoder is designed for the TARGET (y), not feature columns
label_enc = LabelEncoder()
y_demo = ["cat", "dog", "cat", "bird"]
y_encoded = label_enc.fit_transform(y_demo)
print("LabelEncoder on a target:", y_encoded, " classes:", label_enc.classes_)

print("\nIf misused on a FEATURE column instead, a linear/distance-based model would")
print("interpret e.g. bird=0, cat=1, dog=2 as having real numeric order/distance --")
print("implying 'dog' is 'twice as much category' as 'cat', which is meaningless.")
print("OneHotEncoder (unordered) avoids this entirely for feature columns.")


LabelEncoder on a target: [1 2 1 0]  classes: ['bird' 'cat' 'dog']

If misused on a FEATURE column instead, a linear/distance-based model would
interpret e.g. bird=0, cat=1, dog=2 as having real numeric order/distance --
implying 'dog' is 'twice as much category' as 'cat', which is meaningless.
OneHotEncoder (unordered) avoids this entirely for feature columns.


### Common Mistakes
- Using `LabelEncoder` on a feature column instead of `OneHotEncoder`/`OrdinalEncoder` — a historically extremely common mistake (partly because `LabelEncoder` was often the first encoder tutorials introduced); it silently fabricates false ordinal relationships between unordered categories, which linear models and distance-based models will genuinely (and wrongly) use.
- Letting `OrdinalEncoder` default to alphabetical category order for a genuinely ordinal column — always pass `categories=` explicitly.
- Not setting `handle_unknown="ignore"` for a production `OneHotEncoder`, causing inference-time crashes the moment a genuinely new category appears.
- One-hot encoding a very high-cardinality column (thousands of unique values, e.g., a raw ZIP code) without considering dimensionality — direct connection to Chapter 62's sparse matrix chapter.

### Best Practices
- `OneHotEncoder` for unordered categorical features (the default choice); `OrdinalEncoder` with explicit `categories=` only for genuinely ordered features; `LabelEncoder` reserved strictly for the target.
- Always set `handle_unknown="ignore"` for any encoder that will run in a production inference pipeline.
- Inspect `.categories_` after fitting to confirm the encoder learned what you expect.

### Interview Insight
**Q: Why is using `LabelEncoder` on a categorical feature column considered a mistake, even though it "works" without error?**
A: `LabelEncoder` assigns arbitrary integers (0, 1, 2, ...) with no inherent meaning to unordered categories, but many models (linear models via their learned coefficients, KNN/SVM via distance calculations) will treat those integers as having real numeric magnitude and order — implying, for example, that a category encoded as 2 is "twice" a category encoded as 1, or that categories 0 and 2 are "farther apart" than 0 and 1. This is a real, silently-introduced bias with no error message to catch it. `OneHotEncoder` avoids the problem entirely by giving each category its own independent binary column with no implied order or magnitude.

### Mini Exercise
Build a small DataFrame with a genuinely ordinal column (e.g., `education = ["High School", "Bachelors", "Masters", "PhD"]`) and encode it correctly with `OrdinalEncoder(categories=[[...]])`, then deliberately encode it incorrectly by letting `OrdinalEncoder` default to alphabetical order — compare the two resulting integer sequences.

### Quiz (Chapter 6)
1. Which encoder should be used for the target `y`, and which for unordered feature columns?
2. Why must `OrdinalEncoder`'s `categories=` be specified explicitly for genuinely ordinal data?
3. What does `handle_unknown="ignore"` do, and why is it essential for production pipelines?


---
# CHAPTER 7 — MISSING VALUE IMPUTATION

## 1. Concept
`sklearn.impute` fills missing values with a learned or specified substitute, following the exact same `fit`/`transform` contract as scalers — critically, this means **imputation statistics (mean, median, most-frequent value) must also be learned only from training data**, for the identical leakage reason established in Chapter 5.

## 2. The Three Imputers


In [6]:
from sklearn.impute import SimpleImputer, KNNImputer

df_missing = pd.DataFrame({
    "age": [25, np.nan, 35, 40, np.nan, 30],
    "income": [50000, 60000, np.nan, 80000, 55000, np.nan],
    "city": ["NYC", "LA", None, "NYC", "LA", "SF"],
})
print(df_missing)
print("\nMissing count per column:\n", df_missing.isna().sum())


    age   income  city
0  25.0  50000.0   NYC
1   NaN  60000.0    LA
2  35.0      NaN  None
3  40.0  80000.0   NYC
4   NaN  55000.0    LA
5  30.0      NaN    SF

Missing count per column:
 age       2
income    2
city      1
dtype: int64


In [7]:
# SimpleImputer -- the standard default, strategy depends on column type
num_imputer = SimpleImputer(strategy="median")   # median: robust to outliers, unlike mean
age_income_imputed = num_imputer.fit_transform(df_missing[["age", "income"]])
print("Numerical columns imputed with median:")
print(age_income_imputed)

cat_imputer = SimpleImputer(strategy="most_frequent")   # most_frequent: the ONLY valid strategy for categoricals
city_imputed = cat_imputer.fit_transform(df_missing[["city"]])
print("\nCategorical column imputed with most_frequent:")
print(city_imputed.ravel())


Numerical columns imputed with median:
[[2.50e+01 5.00e+04]
 [3.25e+01 6.00e+04]
 [3.50e+01 5.75e+04]
 [4.00e+01 8.00e+04]
 [3.25e+01 5.50e+04]
 [3.00e+01 5.75e+04]]

Categorical column imputed with most_frequent:
['NYC' 'LA' None 'NYC' 'LA' 'SF']


### Parameter Explanation
| `strategy` | Applies to | Notes |
|---|---|---|
| `"mean"` | Numerical | Sensitive to outliers/skew (Matplotlib course Ch 19's lesson applies directly) |
| `"median"` | Numerical | **Preferred default** — robust to outliers and skewed distributions |
| `"most_frequent"` | Numerical or categorical | The only sensible strategy for categorical columns |
| `"constant"` | Either | Fill with a specific value via `fill_value=` (e.g., `"Unknown"` for categoricals, or a sentinel like `-1`) |

### Missingness Indicators — Preserving the "Was Missing" Signal


In [8]:
imputer_with_indicator = SimpleImputer(strategy="median", add_indicator=True)
result_with_indicator = imputer_with_indicator.fit_transform(df_missing[["age", "income"]])
print("Shape WITHOUT indicator would be (6, 2); WITH indicator:", result_with_indicator.shape)
print(result_with_indicator)
print("\nThe extra columns (3rd, 4th) are binary 'was this value missing?' flags --")
print("preserving the missingness signal itself as a feature, exactly the Chapter 69")
print("(Matplotlib course) lesson that missingness can itself be informative.")


Shape WITHOUT indicator would be (6, 2); WITH indicator: (6, 4)
[[2.50e+01 5.00e+04 0.00e+00 0.00e+00]
 [3.25e+01 6.00e+04 1.00e+00 0.00e+00]
 [3.50e+01 5.75e+04 0.00e+00 1.00e+00]
 [4.00e+01 8.00e+04 0.00e+00 0.00e+00]
 [3.25e+01 5.50e+04 1.00e+00 0.00e+00]
 [3.00e+01 5.75e+04 0.00e+00 1.00e+00]]

The extra columns (3rd, 4th) are binary 'was this value missing?' flags --
preserving the missingness signal itself as a feature, exactly the Chapter 69
(Matplotlib course) lesson that missingness can itself be informative.


In [9]:
# KNNImputer -- fills missing values using the average of the K nearest neighbors (by the OTHER features)
knn_imputer = KNNImputer(n_neighbors=2)
knn_result = knn_imputer.fit_transform(df_missing[["age", "income"]])
print("KNNImputer result (uses feature similarity, not just a single global statistic):")
print(knn_result)


KNNImputer result (uses feature similarity, not just a single global statistic):
[[2.50e+01 5.00e+04]
 [3.25e+01 6.00e+04]
 [3.50e+01 6.50e+04]
 [4.00e+01 8.00e+04]
 [3.25e+01 5.50e+04]
 [3.00e+01 6.50e+04]]


### Internal Working
`KNNImputer` finds, for each row with a missing value, the `n_neighbors` most similar **other rows** (by Euclidean distance over the *non-missing* features) and imputes using their average — a more sophisticated approach than `SimpleImputer`'s single global statistic, since it can capture that, e.g., customers similar in income tend to have similar ages, rather than assuming every missing age should get the same overall median regardless of context. This comes at real computational cost (distance computation across the whole dataset) and, like any distance-based method, is scale-sensitive — features should generally be scaled before `KNNImputer`, an important ordering consideration for pipeline construction (Chapter 11).

### `add_indicator=True` — Why This Matters, Concretely
Directly connecting to the Matplotlib course's Chapter 69 lesson: if missingness in a column correlates with the target (e.g., higher earners more often skip reporting income), a plain imputed value discards that signal entirely — but a `MissingIndicator` column preserves it as an explicit, usable feature for the model to learn from.

### Common Mistakes
- Using `strategy="mean"` on a skewed numerical column — same Chapter 19 (Matplotlib course) lesson applied to imputation: the mean is pulled toward the tail, producing an imputed value that doesn't represent a "typical" observation.
- Fitting an imputer on the full dataset before splitting — identical leakage pattern to Chapter 5's scaling mistake; imputation statistics must be learned from training data only.
- Applying `KNNImputer` to unscaled features with wildly different ranges — since it's distance-based, a large-range feature will dominate the neighbor-finding process exactly as in Chapter 5's KNN scaling demonstration.
- Discarding the missingness signal by imputing without `add_indicator=True` when missingness is plausibly informative (Chapter 69, Matplotlib course).

### Best Practices
- Default to `strategy="median"` for numerical columns, `strategy="most_frequent"` for categorical columns.
- Fit imputers on training data only.
- Consider `add_indicator=True` whenever missingness might itself carry signal — cheap to add, and lets the model decide whether the indicator is useful rather than assuming it isn't.
- Scale features before `KNNImputer` if using it.

### Interview Insight
**Q: When would you choose `KNNImputer` over `SimpleImputer`, and what's the tradeoff?**
A: `KNNImputer` captures relationships between features — imputing based on similar rows rather than one global statistic — which can produce more accurate imputations when missingness relates to other observable features (e.g., income correlating with age and education). The tradeoff is computational cost (distance calculations scale with dataset size) and the requirement that features be reasonably scaled first, since it's a distance-based method exactly like KNN classification. For large datasets or when a simple, fast baseline is acceptable, `SimpleImputer` with `strategy="median"` is often good enough and much cheaper.

### Mini Exercise
Add `add_indicator=True` to the `KNNImputer` example above (it supports the same parameter) and inspect the resulting shape to confirm the missingness-flag columns are appended exactly as they were for `SimpleImputer`.

### Quiz (Chapter 7)
1. Why is `strategy="median"` generally preferred over `strategy="mean"` for numerical imputation?
2. What does `add_indicator=True` add, and why might it matter for model performance?
3. Why must `KNNImputer`'s input typically be scaled first?
4. Why must imputation statistics be learned from training data only?


---
# CHAPTER 8 — FEATURE GENERATION

## 1. Concept
`sklearn.preprocessing` also includes transformers that **create new features** from existing ones, rather than just rescaling/re-encoding what's already there — most commonly, polynomial and interaction terms for models (like linear regression) that can't learn non-linear relationships on their own.

## 2. `PolynomialFeatures`


In [10]:
from sklearn.preprocessing import PolynomialFeatures

X_poly_demo = np.array([[2, 3], [4, 5]])   # 2 samples, 2 features: [a, b]

poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly_out = poly.fit_transform(X_poly_demo)

print("Original features: [a, b]")
print(X_poly_demo)
print("\nExpanded features:", poly.get_feature_names_out(["a", "b"]))
print(X_poly_out)


Original features: [a, b]
[[2 3]
 [4 5]]

Expanded features: ['a' 'b' 'a^2' 'a b' 'b^2']
[[ 2.  3.  4.  6.  9.]
 [ 4.  5. 16. 20. 25.]]


### Parameter Explanation
| Parameter | Meaning |
|---|---|
| `degree` | Highest polynomial power to generate (`2` produces squares and pairwise products) |
| `interaction_only` | If `True`, generate ONLY cross-terms (`a*b`), skipping pure powers (`a²`, `b²`) — useful when you specifically believe features *interact* but don't individually have a non-linear effect |
| `include_bias` | If `True`, adds a constant column of 1s (redundant when the downstream model already fits its own intercept — usually set `False`) |

### Why This Matters for Linear Models Specifically


In [11]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

np.random.seed(1)
x_nl = np.linspace(-3, 3, 200).reshape(-1, 1)
y_nl = (x_nl.ravel() ** 2) + np.random.normal(0, 1, 200)   # genuinely QUADRATIC relationship

x_train_nl, x_test_nl, y_train_nl, y_test_nl = train_test_split(x_nl, y_nl, test_size=0.3, random_state=1)

# Plain linear regression -- can only fit a STRAIGHT line to a curved relationship
lin_plain = LinearRegression().fit(x_train_nl, y_train_nl)
r2_plain = r2_score(y_test_nl, lin_plain.predict(x_test_nl))

# Linear regression WITH polynomial features -- can now fit the curve
poly_transform = PolynomialFeatures(degree=2, include_bias=False)
x_train_poly = poly_transform.fit_transform(x_train_nl)
x_test_poly = poly_transform.transform(x_test_nl)
lin_poly = LinearRegression().fit(x_train_poly, y_train_nl)
r2_poly = r2_score(y_test_nl, lin_poly.predict(x_test_poly))

print(f"Plain LinearRegression R^2 (straight line on curved data): {r2_plain:.3f}")
print(f"LinearRegression + PolynomialFeatures(degree=2) R^2:        {r2_poly:.3f}")
print("\nAdding the SQUARED term lets a fundamentally linear model capture a quadratic relationship.")


Plain LinearRegression R^2 (straight line on curved data): -0.109
LinearRegression + PolynomialFeatures(degree=2) R^2:        0.896

Adding the SQUARED term lets a fundamentally linear model capture a quadratic relationship.


### `FunctionTransformer` — Wrapping Arbitrary Functions


In [12]:
from sklearn.preprocessing import FunctionTransformer

log_transformer = FunctionTransformer(np.log1p, validate=True)   # log1p = log(1+x), handles x=0 safely
skewed_data = np.array([[1], [5], [10], [100], [1000]])
log_result = log_transformer.transform(skewed_data)
print("Original:", skewed_data.ravel())
print("Log-transformed (log1p):", log_result.ravel().round(3))
print("\nFunctionTransformer wraps ANY function into the standard Scikit-learn transformer")
print("interface -- so it can slot directly into a Pipeline (Chapter 11) alongside every")
print("other transformer, e.g. for a right-skewed feature like income or transaction amount.")


Original: [   1    5   10  100 1000]
Log-transformed (log1p): [0.693 1.792 2.398 4.615 6.909]

FunctionTransformer wraps ANY function into the standard Scikit-learn transformer
interface -- so it can slot directly into a Pipeline (Chapter 11) alongside every
other transformer, e.g. for a right-skewed feature like income or transaction amount.


### Internal Working
`PolynomialFeatures` generates every combination of input features up to the specified `degree` using combinatorics — for `degree=2` on 2 features `[a, b]`, it produces `[a, b, a², a*b, b²]`; the number of generated features grows **combinatorially** with both the original feature count and the degree, which is why `PolynomialFeatures` is rarely used with `degree > 2` or on datasets with many original features (dimensionality explosion, directly connecting to Chapter 32/34's feature selection and PCA chapters as a common follow-up remedy).

### Common Mistakes
- Applying `PolynomialFeatures(degree=2)` to a dataset with 100 original features — produces thousands of generated features (`~100²/2`), often more than the number of training samples, a direct route to severe overfitting.
- Using `PolynomialFeatures` with tree-based models — largely pointless, since trees can already capture non-linear splits and interactions natively without needing engineered polynomial terms; this technique specifically matters for linear models.
- Fitting `PolynomialFeatures` (or any `FunctionTransformer` with learned state) after splitting incorrectly or on the wrong subset — though `PolynomialFeatures` itself has no learnable statistics dependent on the data's *values* (just its *shape*), it's still good practice to fit only on training data and transform test data separately, matching the pattern established throughout this course.

### Best Practices
- Reserve `PolynomialFeatures` for linear models on datasets with a small-to-moderate number of original features where genuine non-linear/interaction effects are suspected.
- Use `FunctionTransformer` to wrap a right-skew-correcting log transform (or any custom, stateless mathematical transformation) so it participates cleanly in a Pipeline.

### Interview Insight
**Q: Why would you add `PolynomialFeatures` before a `LinearRegression` model, but not before a `RandomForestRegressor`?**
A: Linear regression can only ever fit a linear combination of its input features — it has no way to capture a genuinely curved (e.g., quadratic) relationship unless you explicitly engineer that curvature as an input feature (e.g., `x²`), which is exactly what `PolynomialFeatures` automates. Random Forest (and tree-based models generally) can already represent non-linear relationships and feature interactions natively through their recursive splitting, so adding polynomial features provides little to no benefit and mainly adds redundant dimensionality.

### Mini Exercise
Try `PolynomialFeatures(degree=3, interaction_only=True)` on a 3-feature array and inspect `get_feature_names_out()` to see that only cross-terms (no `a³`, `a²b`, etc.) are generated.

### Quiz (Chapter 8)
1. Why does `PolynomialFeatures(degree=2)` on 100 features risk severe overfitting?
2. What does `interaction_only=True` change about the generated features?
3. Why is `PolynomialFeatures` typically unnecessary for tree-based models?
4. What does `FunctionTransformer` let you do that no other single preprocessing class does?


---
# CHAPTER 9 — CUSTOM TRANSFORMERS

## 1. Concept
When no built-in transformer does what you need (a domain-specific ratio feature, extracting date parts, custom outlier treatment), you write your **own transformer** by subclassing `BaseEstimator` and `TransformerMixin` — this makes it a full first-class citizen of the Scikit-learn ecosystem, usable inside a `Pipeline`/`ColumnTransformer` exactly like any built-in transformer.

## 2. The Required Structure


In [13]:
from sklearn.base import BaseEstimator, TransformerMixin

class RatioFeatureAdder(BaseEstimator, TransformerMixin):
    """Adds a new column: numerator_col / denominator_col."""

    def __init__(self, numerator_col, denominator_col, new_col_name="ratio"):
        # RULE: __init__ must ONLY store arguments as identically-named attributes.
        # No validation, no computation, no logic here -- that all belongs in fit()/transform().
        self.numerator_col = numerator_col
        self.denominator_col = denominator_col
        self.new_col_name = new_col_name

    def fit(self, X, y=None):
        # This transformer has no parameters to LEARN from data -- fit() just returns self.
        return self

    def transform(self, X):
        X = X.copy()   # NEVER mutate the input in place
        X[self.new_col_name] = X[self.numerator_col] / X[self.denominator_col].replace(0, np.nan)
        return X


df_ratio_demo = pd.DataFrame({"debt": [1000, 2000, 1500], "income": [50000, 40000, 0]})
ratio_adder = RatioFeatureAdder(numerator_col="debt", denominator_col="income", new_col_name="debt_to_income")
result_ratio = ratio_adder.fit_transform(df_ratio_demo)
print(result_ratio)


   debt  income  debt_to_income
0  1000   50000            0.02
1  2000   40000            0.05
2  1500       0             NaN


### Line-by-Line Explanation
- **`__init__` rule**: every argument must be stored as an identically-named attribute (`self.numerator_col = numerator_col`), with **zero** other logic. This isn't a style preference — `BaseEstimator.get_params()` inspects `__init__`'s signature directly via `inspect.signature()` to build the parameters dict, so any deviation (renaming, computing something, validating) breaks `get_params()`/`set_params()`, which in turn breaks `GridSearchCV` (Chapter 25) and `clone()` (used internally by cross-validation) for this transformer.
- **`fit(self, X, y=None)`**: must accept `y=None` even if unused — this keeps the signature compatible with how `Pipeline` calls `.fit(X, y)` uniformly across every step, whether or not that particular step actually needs `y`.
- **`transform(self, X)`**: `X = X.copy()` is a critical defensive habit — mutating the caller's original DataFrame in place is a classic source of confusing bugs elsewhere in a larger pipeline/notebook.
- **`TransformerMixin`** automatically provides `.fit_transform()` for free, implemented generically as `self.fit(X, y).transform(X)` — you never need to write `fit_transform()` yourself unless you have a genuine efficiency reason to combine the two steps.

## 3. A More Realistic Example: Date Feature Extraction


In [14]:
class DateFeatureExtractor(BaseEstimator, TransformerMixin):
    """Extracts year, month, day-of-week from a datetime column; drops the original column."""

    def __init__(self, date_col):
        self.date_col = date_col

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        dt = pd.to_datetime(X[self.date_col])
        X[f"{self.date_col}_year"] = dt.dt.year
        X[f"{self.date_col}_month"] = dt.dt.month
        X[f"{self.date_col}_dow"] = dt.dt.dayofweek
        return X.drop(columns=[self.date_col])


df_date_demo = pd.DataFrame({"signup_date": ["2024-01-15", "2024-06-20", "2025-03-01"], "value": [1, 2, 3]})
extractor = DateFeatureExtractor(date_col="signup_date")
print(extractor.fit_transform(df_date_demo))


   value  signup_date_year  signup_date_month  signup_date_dow
0      1              2024                  1                0
1      2              2024                  6                3
2      3              2025                  3                5


In [15]:
# Confirming get_params()/set_params() work correctly BECAUSE __init__ followed the rule
print("get_params():", extractor.get_params())
extractor.set_params(date_col="signup_date")   # would work identically for GridSearchCV tuning this "hyperparameter"
print("\nThis works correctly only because __init__ stored arguments as identically-named attributes.")
print("If we had written self.column_name = date_col (a DIFFERENT name), get_params() would break.")


get_params(): {'date_col': 'signup_date'}

This works correctly only because __init__ stored arguments as identically-named attributes.
If we had written self.column_name = date_col (a DIFFERENT name), get_params() would break.


### Common Mistakes
- Doing validation, computation, or side effects inside `__init__` instead of `fit()`/`transform()` — breaks `get_params()`/`set_params()`/`clone()`, and by extension breaks `GridSearchCV` and cross-validation for this transformer, often with a confusing error far from the actual root cause.
- Renaming an `__init__` argument when storing it (e.g., `self.col = numerator_col` instead of `self.numerator_col = numerator_col`) — same breakage, for the same reason.
- Forgetting `X = X.copy()` in `transform()` — silently mutates the caller's original DataFrame, which can cause confusing bugs elsewhere if that DataFrame is reused.
- Forgetting `y=None` as `fit()`'s second parameter, even when unused — breaks compatibility with `Pipeline.fit(X, y)`'s uniform calling convention.

### Best Practices
- Always inherit from both `BaseEstimator` (for `get_params`/`set_params`) and `TransformerMixin` (for the free `fit_transform`).
- Keep `__init__` a pure, logic-free attribute-storage step.
- Always `.copy()` the input in `transform()`.
- Write custom transformers the moment a specific data-cleaning/feature-engineering step needs to be applied consistently across training and inference — exactly the same "promote to reusable code" instinct from the Matplotlib course's Chapter 87.

### Interview Insight
**Q: You write a custom transformer, but `GridSearchCV` fails with a confusing error when trying to tune one of its parameters. What's the most likely cause?**
A: The `__init__` method likely violates Scikit-learn's convention of storing every argument as an identically-named attribute with no additional logic — `BaseEstimator.get_params()` inspects `__init__`'s signature directly to discover tunable parameters, and any renaming, validation, or computation inside `__init__` breaks that introspection, which in turn breaks `clone()` (used internally during cross-validation) and therefore `GridSearchCV`.

### Mini Exercise
Write a custom transformer `OutlierCapper(BaseEstimator, TransformerMixin)` that, in `fit()`, learns the 1st and 99th percentile of a numerical column from training data (storing them as `self.lower_`, `self.upper_` — trailing underscore, the standard convention for fitted attributes), and in `transform()`, clips any column values beyond those learned bounds.

### Quiz (Chapter 9)
1. What single rule must every custom transformer's `__init__` follow, and what breaks if it's violated?
2. Why must `fit()` accept `y=None` even for a transformer that never uses `y`?
3. What does inheriting from `TransformerMixin` provide for free?
4. Why is `X = X.copy()` important inside `transform()`?


---
# CHAPTER 10 — COLUMNTRANSFORMER

## 1. Concept
Real datasets mix numerical and categorical columns that each need **completely different preprocessing** (scaling vs. encoding). `ColumnTransformer` applies a different transformer (or sub-pipeline of transformers) to different column subsets, then concatenates the results back into a single feature matrix — all within one fittable, `.transform()`-able object.

```
Numerical columns  → Imputer → Scaler   ─┐
                                          ├─→ ColumnTransformer → single combined output → Model
Categorical columns → Imputer → OneHot  ─┘
```

## 2. Building One, Step by Step


In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

df_mixed = pd.DataFrame({
    "age": [25, np.nan, 35, 40, 22],
    "income": [50000, 60000, np.nan, 80000, 45000],
    "city": ["NYC", "LA", "NYC", None, "SF"],
    "plan": ["Basic", "Premium", "Standard", "Basic", "Premium"],
})

numerical_cols = ["age", "income"]
categorical_cols = ["city", "plan"]

# Each branch is its OWN small Pipeline: impute, then transform
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_cols),
    ("cat", categorical_pipeline, categorical_cols),
], remainder="drop")   # "drop" any columns not explicitly listed (the safe, explicit default)

result = preprocessor.fit_transform(df_mixed)
print("Output shape:", result.shape)
print(result)


Output shape: (5, 9)
[[-0.82696125 -0.66208471  0.          1.          0.          0.
   1.          0.          0.        ]
 [-0.06125639  0.16552118  1.          0.          0.          0.
   0.          1.          0.        ]
 [ 0.70444848 -0.24828177  0.          1.          0.          0.
   0.          0.          1.        ]
 [ 1.47015334  1.82073295  0.          0.          0.          1.
   1.          0.          0.        ]
 [-1.28638417 -1.07588766  0.          0.          1.          0.
   0.          1.          0.        ]]


### Parameter Explanation
| Parameter | Meaning |
|---|---|
| `transformers` | A list of `(name, transformer, columns)` 3-tuples — each `columns` argument selects which columns that branch applies to |
| `remainder` | What to do with columns **not** listed in any branch: `"drop"` (default, explicit and safe), `"passthrough"` (keep unchanged), or another transformer to apply to everything else |
| `sparse_threshold` | Controls whether the combined output is a sparse or dense matrix, based on the proportion of sparse input (relevant once `OneHotEncoder` is involved — Chapter 62) |
| `transformer_weights` | Optional per-branch weighting of the output (rarely used) |
| `verbose_feature_names_out` | If `True` (default), prefixes output feature names with the branch name (`num__age`, `cat__city_NYC`) — genuinely useful for later interpretation, occasionally verbose enough to turn off for readability |

### Getting Interpretable Feature Names Back


In [17]:
feature_names = preprocessor.get_feature_names_out()
print("Combined output feature names:")
print(feature_names)

result_df = pd.DataFrame(result, columns=feature_names)
print()
print(result_df)


Combined output feature names:
['num__age' 'num__income' 'cat__city_LA' 'cat__city_NYC' 'cat__city_SF'
 'cat__city_None' 'cat__plan_Basic' 'cat__plan_Premium'
 'cat__plan_Standard']

   num__age  num__income  cat__city_LA  cat__city_NYC  cat__city_SF  \
0 -0.826961    -0.662085           0.0            1.0           0.0   
1 -0.061256     0.165521           1.0            0.0           0.0   
2  0.704448    -0.248282           0.0            1.0           0.0   
3  1.470153     1.820733           0.0            0.0           0.0   
4 -1.286384    -1.075888           0.0            0.0           1.0   

   cat__city_None  cat__plan_Basic  cat__plan_Premium  cat__plan_Standard  
0             0.0              1.0                0.0                 0.0  
1             0.0              0.0                1.0                 0.0  
2             0.0              0.0                0.0                 1.0  
3             1.0              1.0                0.0                 0.0  
4         

### Line-by-Line Explanation
- `("num", numerical_pipeline, numerical_cols)` — the 3-tuple's `name` (`"num"`) becomes a prefix in `get_feature_names_out()`'s output, letting you trace every final column back to which branch produced it — essential once you have feature importances or coefficients to interpret (Chapter 59).
- Each branch is itself a full `Pipeline` (imputer → scaler, or imputer → encoder) — `ColumnTransformer` doesn't require single transformers per branch; nesting Pipelines inside a ColumnTransformer inside an outer Pipeline (Chapter 11) is the standard, fully general pattern for real preprocessing.
- `remainder="drop"` — any column not explicitly assigned to a branch is silently dropped. This is deliberately **explicit and safe**: a column accidentally left out of both `numerical_cols` and `categorical_cols` won't crash anything, but also won't silently leak into the model unprocessed and unnoticed — always double-check `.get_feature_names_out()` against your expected column list to catch this kind of omission.

### `make_column_transformer` — The Shorthand Constructor


In [18]:
from sklearn.compose import make_column_transformer

# Equivalent to the explicit ColumnTransformer above, but auto-generates branch names
preprocessor_v2 = make_column_transformer(
    (numerical_pipeline, numerical_cols),
    (categorical_pipeline, categorical_cols),
    remainder="drop",
)
result_v2 = preprocessor_v2.fit_transform(df_mixed)
print("make_column_transformer produces identical results:", np.allclose(result, result_v2))
print("Auto-generated branch names:", [name for name, _, _ in preprocessor_v2.transformers])


make_column_transformer produces identical results: True
Auto-generated branch names: ['pipeline-1', 'pipeline-2']


### Common Mistakes
- Forgetting a column entirely (not listed in any branch, and `remainder="drop"`) — the column silently disappears from the model's input with no error, a genuinely dangerous silent failure mode; always cross-check `get_feature_names_out()` against your expected full column list.
- Using `remainder="passthrough"` carelessly on a column that still needs preprocessing (e.g., a raw unscaled numerical column) — it passes straight through unprocessed into the model.
- Applying a `ColumnTransformer` fit on training data's specific column set to a new dataset missing one of those columns — raises a clear error, but worth anticipating in production input validation (Chapter 80).
- Selecting columns by **position** (integer indices) instead of by **name** when working with a DataFrame — fragile if column order ever changes upstream; prefer name-based selection.

### Best Practices
- Always inspect `.get_feature_names_out()` after building a `ColumnTransformer`, both to verify correctness and to have interpretable names ready for later feature importance work.
- Default to `remainder="drop"` (explicit) over `"passthrough"` (implicit) unless you specifically intend every other column to flow through completely unprocessed.
- Nest full `Pipeline`s as branches (impute → scale/encode) rather than single bare transformers, since real columns almost always need more than one preprocessing step.

### Interview Insight
**Q: You have a dataset with numerical and categorical columns needing entirely different preprocessing. How do you handle this in one unified, fittable object?**
A: `ColumnTransformer`, with one branch per column group — typically a small `Pipeline` per branch (e.g., `SimpleImputer` → `StandardScaler` for numerical columns; `SimpleImputer` → `OneHotEncoder` for categorical columns) — combined via `ColumnTransformer([("num", num_pipeline, num_cols), ("cat", cat_pipeline, cat_cols)])`. This produces one object with a single `.fit_transform()`/`.transform()` call that correctly routes each column subset through its appropriate preprocessing and concatenates the results, ready to feed directly into a model (and, per Chapter 11, into the model as one more step of an outer `Pipeline`).

### Mini Exercise
Add a third numerical column to `df_mixed` (e.g., `"tenure_months"`) without adding it to `numerical_cols`, refit the `ColumnTransformer`, and confirm (via `get_feature_names_out()`) that it silently disappeared from the output — directly demonstrating this chapter's "forgotten column" warning.

### Quiz (Chapter 10)
1. What does each branch of a `ColumnTransformer` consist of (name, transformer, columns) — what can the "transformer" position actually be?
2. What are the three options for `remainder=`, and which is the safer default?
3. Why is `get_feature_names_out()` essential to check after building a `ColumnTransformer`?
4. What silent failure mode does an accidentally-omitted column produce with `remainder="drop"`?

### Chapter 10 Summary
- `ColumnTransformer` is how real, mixed-type datasets get preprocessed correctly in one unified object.
- Each branch is typically its own mini-`Pipeline` (impute → scale/encode), combined via `ColumnTransformer([(name, pipeline, columns), ...])`.
- Always verify `get_feature_names_out()` against your expected columns — the single most important habit this chapter establishes.

---

## END OF NOTEBOOK 2 — What You Should Now Be Able To Do
- Correctly encode categorical features (OneHot for unordered, Ordinal with explicit order for ordered, LabelEncoder reserved for the target only) and handle unseen categories safely in production.
- Impute missing values with the correct strategy per column type, and know when a missingness indicator or KNN-based imputation adds real value.
- Generate polynomial/interaction features for linear models and understand exactly why tree-based models don't need this.
- Write a fully-compliant custom transformer that correctly integrates with `get_params()`/`set_params()`/`GridSearchCV`.
- Build a complete `ColumnTransformer` routing numerical and categorical columns through separate, appropriate preprocessing pipelines, and verify its output with `get_feature_names_out()`.

## Before Moving to Notebook 3 (Chapters 11–15: Pipelines, FeatureUnion/Advanced Composition, Supervised Learning Architecture, Linear Regression Family, Logistic Regression)

